### Chủ đề số 14: Bank marketing
Bài toán: Dự đoán khách hàng có đăng ký gửi tiền kỳ hạn hay không (Yes/No)
Loại bài toán: Classification

Tập dữ liệu: UCI Bank Marketing (https://archive.ics.uci.edu/dataset/222/bank+marketing)
Sử dụng 4 thuật toán: Logistic Regression, KNN, Decision Tree, Random Forest
Độ đo: F1-score, Recall, ROC-AUC

#### 1. Giới thiệu bài toán và dữ liệu
1.1. Bài toán
- Bối cảnh: dữ liệu ghi lại các cuộc gọi marketing trực tiếp của một ngân hàng nhằm mời khách hàng gửi tiền kỳ hạn.
- Mục tiêu: dự đoán khách hàng có đăng ký tiền gửi kỳ hạn hay không, giúp ngân hàng ưu tiên liên hệ nhóm khách hàng tiềm năng và giảm chi phí cuộc gọi.
- Input: thông tin khách hàng và chiến dịch marketing như tuổi, nghề nghiệp, lịch sử liên hệ và chỉ số kinh tế vĩ mô.
- Output: yes (đăng ký) hoặc no (không đăng ký).
- Loại bài toán: học có giám sát, phân loại nhị phân (Binary Classification).

#### 1.2. Dữ liệu
Nguồn: UCI Bank Marketing, tệp bank-additional-full.csv, gồm 41.188 mẫu và 21 cột (20 thuộc tính và biến mục tiêu y).
Nhóm thuộc tính	Các cột
Thông tin khách hàng	age, job, marital, education, default, housing, loan
Thông tin lần liên hệ hiện tại	contact, month, day_of_week, duration
Lịch sử chiến dịch	campaign, pdays, previous, poutcome
Chỉ số kinh tế vĩ mô	emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed
Biến mục tiêu	y
| Nhóm thuộc tính | Các cột |
|---|---|
| Thông tin khách hàng | `age`, `job`, `marital`, `education`, `default`, `housing`, `loan` |
| Thông tin lần liên hệ hiện tại | `contact`, `month`, `day_of_week`, `duration` |
| Lịch sử chiến dịch | `campaign`, `pdays`, `previous`, `poutcome` |
| Chỉ số kinh tế vĩ mô | `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` |
| Biến mục tiêu | `y` |

Cột duration (thời lượng cuộc gọi) chỉ biết sau khi cuộc gọi kết thúc nên được loại bỏ ở bước tiền xử lý để tránh rò rỉ thông tin (data leakage).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Tìm thư mục gốc của đồ án, để notebook chạy được khi mở từ thư mục riêng
current_dir = Path.cwd().resolve()
ROOT_DIR = next(
    (folder for folder in [current_dir, *current_dir.parents]
     if (folder / "src" / "config.py").exists()),
    None,
)

if ROOT_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy src/config.py. Hãy mở notebook bên trong thư mục Đồ án cuối kỳ."
    )

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import PROCESSED_DATA_DIR, RESULTS_DIR

# Đọc đúng các tập dữ liệu đã tiền xử lý và chia sẵn dùng chung cho cả nhóm
X_train_df = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test_df = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze("columns")

X_train = X_train_df.to_numpy()
X_test = X_test_df.to_numpy()
feature_names = X_train_df.columns.to_numpy()

# Đảm bảo nhãn mục tiêu ở dạng số 0/1 (0 = No, 1 = Yes)
if y_train.dtype == "object":
    y_train = y_train.map({"no": 0, "yes": 1}).astype(int)
if y_test.dtype == "object":
    y_test = y_test.map({"no": 0, "yes": 1}).astype(int)

y_train = pd.Series(y_train).astype(int)
y_test = pd.Series(y_test).astype(int)

# Dùng 5-fold như notebook Random Forest để việc đánh giá CV nhất quán
N_FOLDS = 5
RANDOM_STATE = 42

print("Thư mục dự án:", ROOT_DIR)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Số đặc trưng:", len(feature_names))
print("Phân bố lớp train:")
print(y_train.value_counts().sort_index().rename(index={0: "No", 1: "Yes"}))

In [ ]:
import json
import os
import joblib
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay,
)


def save_experiment(exp_name, model, params, y_true, y_pred, y_proba, root=RESULTS_DIR):
    """Lưu thông số, độ đo, dự đoán, mô hình và biểu đồ của một thí nghiệm."""
    folder = os.path.join(root, exp_name)
    os.makedirs(folder, exist_ok=True)

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
    }

    with open(os.path.join(folder, "params_cfg.json"), "w", encoding="utf-8") as f:
        json.dump(params, f, ensure_ascii=False, indent=2, default=str)
    with open(os.path.join(folder, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    pd.DataFrame([metrics]).round(4).to_csv(
        os.path.join(folder, "metrics.csv"), index=False
    )
    pd.DataFrame({
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "y_proba": np.asarray(y_proba),
    }).to_csv(os.path.join(folder, "predictions.csv"), index=False)
    joblib.dump(model, os.path.join(folder, "best_model.pkl"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, display_labels=["No", "Yes"], cmap="Blues", ax=axes[0]
    )
    axes[0].set_title(f"{exp_name} - Confusion Matrix")
    RocCurveDisplay.from_predictions(y_true, y_proba, ax=axes[1])
    axes[1].plot([0, 1], [0, 1], "k--")
    axes[1].set_title(f"{exp_name} - ROC Curve")
    plt.tight_layout()
    fig.savefig(os.path.join(folder, "confusion_roc.png"), dpi=150)
    plt.close(fig)

    print("Đã lưu thí nghiệm vào:", folder)
    return metrics

### 4. Thuật toán Logistic Regression
#### 4.1. Giới thiệu thuật toán
Logistic Regression (Hồi quy Logistic) là thuật toán học có giám sát được sử dụng cho bài toán phân loại. Mô hình tính tổng có trọng số của các đặc trưng rồi đưa kết quả qua hàm Sigmoid để thu được xác suất khách hàng đăng ký tiền gửi kỳ hạn.
\[
P(y=1\mid X)=\frac{1}{1+e^{-z}},\qquad z=b+w_1x_1+\cdots+w_nx_n
\]
Nếu xác suất dự đoán lớn hơn hoặc bằng ngưỡng 0,5, mô hình dự đoán yes; ngược lại dự đoán no.
- Ưu điểm: huấn luyện tương đối nhanh, dễ giải thích thông qua hệ số, có thể trả về xác suất dự đoán.
- Nhược điểm: ranh giới phân loại cơ bản là tuyến tính; nhạy với thang đo của đặc trưng nên cần chuẩn hóa dữ liệu. Kết quả cũng bị ảnh hưởng bởi mất cân bằng lớp.
Cấu hình baseline: max_iter=2000 để hỗ trợ thuật toán hội tụ, class_weight="balanced" để tăng trọng số cho lớp yes đang chiếm tỷ lệ nhỏ và random_state=42 để thực nghiệm có thể lặp lại.

#### 4.2. Huấn luyện và dự đoán
Huấn luyện mô hình trên tập train đã được tiền xử lý, sau đó dự đoán nhãn và xác suất khách hàng đăng ký (lớp yes) trên tập test.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Khởi tạo mô hình Logistic Regression baseline
model_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

# Huấn luyện trên tập train
model_lr.fit(X_train, y_train)

# Lấy đúng cột xác suất tương ứng với lớp 1 (Yes)
yes_index_lr = list(model_lr.classes_).index(1)
y_pred_lr = model_lr.predict(X_test)
y_proba_lr = model_lr.predict_proba(X_test)[:, yes_index_lr]

print("Huấn luyện và dự đoán Logistic Regression hoàn tất!")
print("\n10 nhãn dự đoán đầu tiên:", y_pred_lr[:10])
print("10 xác suất dự đoán đầu tiên:", y_proba_lr[:10].round(4))

# Lưu kết quả thực nghiệm baseline theo cùng cấu trúc Random Forest
params_lr_baseline = model_lr.get_params()
metrics_lr_baseline = save_experiment(
    "Experiment_04_LogisticRegression_baseline",
    model_lr,
    params_lr_baseline,
    y_test,
    y_pred_lr,
    y_proba_lr,
)

#### 4.3. Đánh giá mô hình
Đánh giá trên tập test bằng Accuracy, Precision, Recall, F1-score và ROC-AUC. Do dữ liệu mất cân bằng, cần chú ý F1-score và Recall thay vì chỉ nhìn Accuracy.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, zero_division=0)
recall_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)
auc_lr = roc_auc_score(y_test, y_proba_lr)

print("===== LOGISTIC REGRESSION BASELINE =====")
print(f"Accuracy : {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall   : {recall_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"ROC-AUC  : {auc_lr:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    y_test,
    y_pred_lr,
    labels=[0, 1],
    target_names=["No", "Yes"],
    zero_division=0,
))

#### 4.4. Trực quan hóa kết quả
Confusion Matrix cho biết số khách hàng được dự đoán đúng/sai ở từng lớp, đặc biệt là số khách hàng thật sự đăng ký nhưng bị bỏ sót (False Negative). ROC Curve thể hiện khả năng phân biệt hai lớp ở nhiều ngưỡng phân loại.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_lr,
    labels=[0, 1],
    display_labels=["No", "Yes"],
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("Logistic Regression - Confusion Matrix")

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_lr,
    name="Logistic Regression",
    ax=axes[1],
)
axes[1].plot([0, 1], [0, 1], "k--", label="Random baseline")
axes[1].set_title("Logistic Regression - ROC Curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

#### 4.5. Nhận xét và lưu kết quả
Nhận xét (cập nhật theo số liệu thực tế sau khi chạy notebook):
- Ghi lại Accuracy, Precision, Recall, F1-score và ROC-AUC của Logistic Regression baseline.
- class_weight="balanced" giúp mô hình chú ý hơn đến lớp yes ít mẫu. Thông thường Recall có thể tăng, nhưng Precision có thể giảm do phát sinh nhiều dự đoán dương tính giả.
- Accuracy không nên được dùng đơn lẻ vì dữ liệu có tỷ lệ lớp no áp đảo. Cần xem đồng thời F1-score, Recall và ROC-AUC.
- Hướng cải thiện: thử SMOTE, điều chỉnh ngưỡng phân loại và tinh chỉnh tham số C.

In [ ]:
# Danh sách dùng để tổng hợp kết quả các phiên bản Logistic Regression
if "all_results" not in globals():
    all_results = []
if "all_probas" not in globals():
    all_probas = {}

# Xóa dòng cũ nếu chạy lại cell để tránh trùng kết quả
all_results[:] = [
    row for row in all_results
    if row.get("Model") != "Logistic Regression (baseline)"
]
all_results.append({
    "Model": "Logistic Regression (baseline)",
    "Accuracy": accuracy_lr,
    "Precision": precision_lr,
    "Recall": recall_lr,
    "F1-score": f1_lr,
    "ROC-AUC": auc_lr,
})
all_probas["Logistic Regression (baseline)"] = y_proba_lr

print("Đã lưu kết quả Logistic Regression baseline!")

### 9. Xử lý mất cân bằng lớp và chỉnh ngưỡng

#### 9.1. Logistic Regression

##### 9.1.1. Cân bằng dữ liệu bằng SMOTE
SMOTE tạo thêm mẫu thuộc lớp thiểu số dựa trên các láng giềng gần. Chỉ áp dụng trên tập train; tập test được giữ nguyên phân bố ban đầu để đánh giá đúng thực tế. Vì tập train sau SMOTE đã được cân bằng nên Logistic Regression dùng class_weight=None, tránh bù trọng số hai lần.

Lưu ý: dữ liệu đã One-Hot Encoding nên các mẫu nội suy có thể chứa giá trị thập phân ở các cột nhị phân.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def lr_metrics(y_true, y_pred, y_proba):
    """Tính 5 độ đo; lớp yes = 1 là lớp dương."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
    }


def save_result(name, metrics, proba):
    """Lưu/cập nhật kết quả dùng chung, không tạo dòng trùng khi chạy lại."""
    global all_results, all_probas
    if "all_results" not in globals():
        all_results = []
    if "all_probas" not in globals():
        all_probas = {}
    all_results[:] = [row for row in all_results if row.get("Model") != name]
    all_results.append({"Model": name, **metrics})
    all_probas[name] = np.asarray(proba)

# Cân bằng dữ liệu trên tập train, không tác động đến tập test
smote_lr = SMOTE(random_state=RANDOM_STATE)
X_train_smote_lr, y_train_smote_lr = smote_lr.fit_resample(X_train, y_train)

print("Phân bố lớp trước SMOTE:", dict(pd.Series(y_train).value_counts()))
print("Phân bố lớp sau SMOTE :", dict(pd.Series(y_train_smote_lr).value_counts()))

# Huấn luyện Logistic Regression trên tập train sau SMOTE
model_lr_smote = LogisticRegression(
    max_iter=2000,
    class_weight=None,
    random_state=RANDOM_STATE,
)
model_lr_smote.fit(X_train_smote_lr, y_train_smote_lr)

yes_index_lr_smote = list(model_lr_smote.classes_).index(1)
y_pred_lr_smote = model_lr_smote.predict(X_test)
y_proba_lr_smote = model_lr_smote.predict_proba(X_test)[:, yes_index_lr_smote]

metrics_lr_smote = lr_metrics(y_test, y_pred_lr_smote, y_proba_lr_smote)
print("\n===== LOGISTIC REGRESSION + SMOTE =====")
for metric, value in metrics_lr_smote.items():
    print(f"{metric:<10}: {value:.4f}")

save_result("Logistic Regression (SMOTE)", metrics_lr_smote, y_proba_lr_smote)
save_experiment(
    "Experiment_08_LogisticRegression_SMOTE",
    model_lr_smote,
    model_lr_smote.get_params(),
    y_test,
    y_pred_lr_smote,
    y_proba_lr_smote,
)

##### 9.1.2. Chỉnh ngưỡng phân loại
Mặc định, Logistic Regression dự đoán yes khi xác suất từ 0,5 trở lên. Với dữ liệu mất cân bằng, ngưỡng 0,5 chưa chắc tối ưu cho F1-score.

Để tránh rò rỉ thông tin, ngưỡng được chọn trên tập train bằng dự đoán chéo (cross-validation 5-fold). Mỗi mẫu train được dự đoán bởi mô hình chưa từng học mẫu đó. Sau đó chọn ngưỡng cho F1-score cao nhất và chỉ áp dụng ngưỡng đã chọn lên tập test.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

skf_lr = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Xác suất OOF trên tập train bằng mô hình baseline
oof_proba_lr = cross_val_predict(
    clone(model_lr),
    X_train,
    y_train,
    cv=skf_lr,
    method="predict_proba",
    n_jobs=-1,
)[:, yes_index_lr]

# Quét ngưỡng tương tự notebook Random Forest: từ 0,05 đến 0,60
thresholds_lr = np.arange(0.05, 0.65, 0.05)
rows_lr = []
for threshold in thresholds_lr:
    pred_oof = (oof_proba_lr >= threshold).astype(int)
    rows_lr.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(y_train, pred_oof, zero_division=0),
        "Recall": recall_score(y_train, pred_oof, zero_division=0),
        "F1-score": f1_score(y_train, pred_oof, zero_division=0),
    })

df_thr_lr = pd.DataFrame(rows_lr)
display(df_thr_lr.round(4))

best_threshold_lr = float(
    df_thr_lr.loc[df_thr_lr["F1-score"].idxmax(), "Threshold"]
)
print(f"Ngưỡng có F1-score cao nhất trên train: {best_threshold_lr:.2f}")

# Biểu đồ Precision, Recall, F1 theo ngưỡng
plt.figure(figsize=(8, 5))
for metric in ["Precision", "Recall", "F1-score"]:
    plt.plot(df_thr_lr["Threshold"], df_thr_lr[metric], marker="o", label=metric)
plt.axvline(best_threshold_lr, color="gray", linestyle="--", label="Ngưỡng được chọn")
plt.axvline(0.5, color="red", linestyle=":", label="Ngưỡng mặc định 0.5")
plt.title("Logistic Regression - Độ đo theo ngưỡng (train, CV 5-fold)")
plt.xlabel("Ngưỡng")
plt.ylabel("Giá trị")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Dùng ngưỡng đã chọn từ train để phân loại xác suất baseline trên test
y_pred_lr_thr = (y_proba_lr >= best_threshold_lr).astype(int)
metrics_lr_thr = lr_metrics(y_test, y_pred_lr_thr, y_proba_lr)

print(f"===== LOGISTIC REGRESSION + NGƯỠNG {best_threshold_lr:.2f} =====")
for metric, value in metrics_lr_thr.items():
    print(f"{metric:<10}: {value:.4f}")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_lr_thr,
    labels=[0, 1],
    display_labels=["No", "Yes"],
    cmap="Blues",
)
plt.title(f"Logistic Regression - Confusion Matrix (threshold={best_threshold_lr:.2f})")
plt.tight_layout()
plt.show()

save_result("Logistic Regression (threshold)", metrics_lr_thr, y_proba_lr)
save_experiment(
    "Experiment_12_LogisticRegression_threshold",
    model_lr,
    {**model_lr.get_params(), "threshold": best_threshold_lr},
    y_test,
    y_pred_lr_thr,
    y_proba_lr,
)

##### 9.1.3. Nhận xét
Nhận xét (điền sau khi chạy, dựa trên số liệu thật):
- Baseline: Recall = ..., F1-score = ... (xem mục 4).
- SMOTE: Recall = ..., Precision = ..., F1-score = ... So với baseline, Recall tăng/giảm ..., ROC-AUC tăng/giảm ...
- Chỉnh ngưỡng: ngưỡng được chọn = ..., Recall = ..., Precision = ..., F1-score = ... Ngưỡng thấp hơn 0,5 giúp phát hiện thêm khách hàng tiềm năng nhưng có thể làm tăng số khách hàng bị dự đoán nhầm là yes.
- Cần cân nhắc chi phí bỏ sót khách hàng thực sự đăng ký (FN) và chi phí liên hệ nhầm (FP) khi lựa chọn ngưỡng phù hợp.

### 10. Tinh chỉnh siêu tham số

#### 10.1. Logistic Regression

##### 10.1.1. Tìm kiếm siêu tham số
Sử dụng GridSearchCV với StratifiedKFold để thử các cấu hình trên tập train. Tối ưu theo F1-score, đồng thời theo dõi Recall và ROC-AUC.
| Tham số | Giá trị thử | Ý nghĩa |
|---|---|---|
| `C` | 0.01, 0.1, 1, 10, 100 | Nghịch đảo độ mạnh regularization; C nhỏ regularization mạnh hơn |
| `class_weight` | `None`, `balanced` | Có/không tăng trọng số cho lớp thiểu số |
| `solver` | `lbfgs` | Thuật toán tối ưu dùng để tìm hệ số |
| `max_iter` | 2000 | Số vòng lặp tối đa để mô hình hội tụ |

In [ ]:
import time
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid_lr = {
    "C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "class_weight": [None, "balanced"],
}

cv_lr = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search_lr = GridSearchCV(
    estimator=LogisticRegression(
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
    param_grid=param_grid_lr,
    scoring={"f1": "f1", "recall": "recall", "roc_auc": "roc_auc"},
    refit="f1",
    cv=cv_lr,
    n_jobs=-1,
    verbose=1,
    error_score="raise",
)

start_lr = time.time()
search_lr.fit(X_train, y_train)
print(f"Thời gian tìm kiếm: {(time.time() - start_lr) / 60:.2f} phút")

print("\nTham số tốt nhất:")
for key, value in search_lr.best_params_.items():
    print(f"  {key}: {value}")
print(f"F1-score CV tốt nhất: {search_lr.best_score_:.4f}")

cv_lr_results = pd.DataFrame(search_lr.cv_results_)
cols_lr = ["rank_test_f1", "mean_test_f1", "std_test_f1", "mean_test_recall", "mean_test_roc_auc", "params"]
display(
    cv_lr_results.sort_values("rank_test_f1")[cols_lr].head(10).round(4)
)

##### 10.1.2. Đánh giá mô hình sau tinh chỉnh
Sử dụng mô hình tốt nhất được GridSearchCV chọn (đã fit lại trên toàn bộ tập train) để dự đoán và đánh giá trên tập test.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay

model_lr_tuned = search_lr.best_estimator_
y_pred_lr_tuned = model_lr_tuned.predict(X_test)
yes_index_lr_tuned = list(model_lr_tuned.classes_).index(1)
y_proba_lr_tuned = model_lr_tuned.predict_proba(X_test)[:, yes_index_lr_tuned]

metrics_lr_tuned = lr_metrics(y_test, y_pred_lr_tuned, y_proba_lr_tuned)

print("===== LOGISTIC REGRESSION (TUNED) =====")
for metric, value in metrics_lr_tuned.items():
    print(f"{metric:<10}: {value:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    y_test,
    y_pred_lr_tuned,
    labels=[0, 1],
    target_names=["No", "Yes"],
    zero_division=0,
))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr_tuned, labels=[0, 1],
    display_labels=["No", "Yes"], cmap="Blues", ax=axes[0]
)
axes[0].set_title("Logistic Regression (tuned) - Confusion Matrix")
RocCurveDisplay.from_predictions(
    y_test, y_proba_lr_tuned, name="Logistic Regression (tuned)", ax=axes[1]
)
axes[1].plot([0, 1], [0, 1], "k--", label="Random baseline")
axes[1].set_title("Logistic Regression (tuned) - ROC Curve")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

save_result("Logistic Regression (tuned)", metrics_lr_tuned, y_proba_lr_tuned)
save_experiment(
    "Experiment_16_LogisticRegression_tuned",
    model_lr_tuned,
    search_lr.best_params_,
    y_test,
    y_pred_lr_tuned,
    y_proba_lr_tuned,
)

##### 10.1.3. Nhận xét
Nhận xét (cập nhật sau khi chạy):
- Tham số tốt nhất: C = ..., class_weight = ...
- So với baseline: F1-score thay đổi ..., Recall thay đổi ..., ROC-AUC thay đổi ...
- C nhỏ làm regularization mạnh hơn, giúp hạn chế hệ số quá lớn; C lớn cho phép mô hình khớp dữ liệu mạnh hơn và có nguy cơ overfitting.
- Kết luận cấu hình tuned có cải thiện hay không dựa trên kết quả test, không chỉ dựa vào F1-score CV

### 11. So sánh trước và sau tinh chỉnh
Tổng hợp bốn phiên bản Logistic Regression trên cùng tập test:
1. Baseline (class_weight="balanced", ngưỡng 0,5)
2. SMOTE (cân bằng lớp trên tập train)
3. Threshold Tuning (chọn ngưỡng tốt nhất bằng CV trên train)
4. Hyperparameter Tuning (GridSearchCV, tối ưu F1-score)

In [ ]:
df_lr_compare = pd.DataFrame([
    {"Model": "Logistic Regression (baseline)", **lr_metrics(y_test, y_pred_lr, y_proba_lr)},
    {"Model": "Logistic Regression (SMOTE)", **metrics_lr_smote},
    {"Model": "Logistic Regression (threshold)", **metrics_lr_thr},
    {"Model": "Logistic Regression (tuned)", **metrics_lr_tuned},
]).set_index("Model")

display(df_lr_compare.round(4))

# Biểu đồ cột so sánh 5 độ đo
metrics_cols_lr = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
ax = df_lr_compare[metrics_cols_lr].plot(kind="bar", figsize=(12, 5))
ax.set_title("So sánh các phiên bản Logistic Regression")
ax.set_ylabel("Giá trị")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=12, ha="right")
plt.legend(loc="lower right", ncol=2)
plt.tight_layout()
plt.show()

# So sánh ROC Curve của các phiên bản
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
proba_versions_lr = {
    "Baseline": y_proba_lr,
    "SMOTE": y_proba_lr_smote,
    "Threshold (same baseline probability)": y_proba_lr,
    "Tuned": y_proba_lr_tuned,
}
for name, probabilities in proba_versions_lr.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc_value = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_value:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random baseline")
plt.title("ROC Curve - Các phiên bản Logistic Regression")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Bảng so sánh phục vụ báo cáo
df_lr_compare.to_csv(
    Path(RESULTS_DIR) / "LogisticRegression_comparison.csv",
    encoding="utf-8-sig",
)
print("Đã lưu bảng so sánh vào:", Path(RESULTS_DIR) / "LogisticRegression_comparison.csv")

### 12. Phân tích mô hình

#### 12.1. Phân tích hệ số (Model Coefficients)
Khác với Random Forest dùng feature_importances_, Logistic Regression có thể giải thích mô hình thông qua hệ số coef_:
- Hệ số dương: đặc trưng tăng làm log-odds (log tỷ số khả năng) thuộc lớp yes tăng, khi các yếu tố khác giữ nguyên.
- Hệ số âm: đặc trưng tăng làm log-odds thuộc lớp yes giảm.
- Độ lớn tuyệt đối: hệ số có trị tuyệt đối càng lớn thì mức liên hệ tuyến tính với dự đoán càng mạnh trong thang đo hiện tại.
So sánh Top 10 hệ số có trị tuyệt đối lớn nhất giữa Logistic Regression Baseline và Tuned. Không xem hệ số là quan hệ nhân quả.

In [ ]:
# Tạo Series hệ số, gắn đúng tên đặc trưng đã tiền xử lý
coef_lr_base = pd.Series(model_lr.coef_.ravel(), index=feature_names)
coef_lr_tuned = pd.Series(model_lr_tuned.coef_.ravel(), index=feature_names)

top10_coef_base = coef_lr_base.reindex(
    coef_lr_base.abs().sort_values(ascending=False).head(10).index
)
top10_coef_tuned = coef_lr_tuned.reindex(
    coef_lr_tuned.abs().sort_values(ascending=False).head(10).index
)

print("Top 10 hệ số theo trị tuyệt đối - Logistic Regression Baseline:")
display(top10_coef_base.to_frame("Coefficient").round(4))
print("Top 10 hệ số theo trị tuyệt đối - Logistic Regression Tuned:")
display(top10_coef_tuned.to_frame("Coefficient").round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top10_coef_base.sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Logistic Regression (Baseline) - Top 10 Coefficients")
axes[0].set_xlabel("Coefficient")
top10_coef_tuned.sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Logistic Regression (Tuned) - Top 10 Coefficients")
axes[1].set_xlabel("Coefficient")
plt.tight_layout()
plt.show()

#### 12.2. Phân tích lỗi (Error Analysis)
Phân tích các dự đoán sai của mô hình tốt nhất (Logistic Regression Tuned) trên tập test:
- False Negative (FN): khách hàng thực tế đăng ký (y=1) nhưng mô hình dự đoán 0 — bỏ lỡ khách hàng tiềm năng.
- False Positive (FP): khách hàng thực tế không đăng ký (y=0) nhưng mô hình dự đoán 1 — ngân hàng tốn chi phí liên hệ không cần thiết.

In [ ]:
from sklearn.metrics import confusion_matrix

# Ưu tiên dùng dữ liệu gốc nếu project đã lưu X_test_raw.csv;
# nếu không thì dùng dữ liệu đã mã hóa/chuẩn hóa để vẫn phân tích được.
raw_test_path = PROCESSED_DATA_DIR / "X_test_raw.csv"
if raw_test_path.exists():
    df_err_lr = pd.read_csv(raw_test_path)
else:
    df_err_lr = X_test_df.copy()

df_err_lr = df_err_lr.reset_index(drop=True)
df_err_lr["y_true"] = y_test.reset_index(drop=True)
df_err_lr["y_pred"] = pd.Series(y_pred_lr_tuned).reset_index(drop=True)
df_err_lr["y_proba"] = pd.Series(y_proba_lr_tuned).reset_index(drop=True)

tn, fp, fn, tp = confusion_matrix(
    df_err_lr["y_true"], df_err_lr["y_pred"], labels=[0, 1]
).ravel()
print(f"TP (Đoán đúng Yes): {tp} | FN (Bỏ sót Yes): {fn}")
print(f"TN (Đoán đúng No) : {tn} | FP (Đoán nhầm Yes): {fp}")

def classify_error_lr(row):
    if row["y_true"] == 1 and row["y_pred"] == 1:
        return "TP"
    if row["y_true"] == 0 and row["y_pred"] == 0:
        return "TN"
    if row["y_true"] == 0 and row["y_pred"] == 1:
        return "FP"
    return "FN"

df_err_lr["error_type"] = df_err_lr.apply(classify_error_lr, axis=1)

# Nếu dữ liệu gốc có các cột số này, so sánh trung bình theo nhóm lỗi
numeric_cols_lr = [
    col for col in ["age", "campaign", "pdays", "previous", "emp.var.rate", "euribor3m", "nr.employed"]
    if col in df_err_lr.columns
]
if numeric_cols_lr:
    print("\nTrung bình đặc trưng số theo nhóm dự đoán:")
    display(df_err_lr.groupby("error_type")[numeric_cols_lr + ["y_proba"]].mean().round(3))

print("\nTop 5 False Negative có xác suất dự đoán thấp nhất:")
display(
    df_err_lr[df_err_lr["error_type"] == "FN"]
    .sort_values("y_proba")
    .head(5)
)

Nhận xét phân tích lỗi:
- Ghi số lượng FN và FP thực tế từ kết quả chạy.
- FN là các khách hàng thực sự đăng ký nhưng bị mô hình bỏ sót; FP là các khách hàng bị dự đoán nhầm là sẽ đăng ký.
- Nếu chi phí bỏ sót khách tiềm năng cao hơn chi phí gọi nhầm, ngân hàng có thể cân nhắc ngưỡng thấp hơn 0,5. Tuy nhiên, cần dựa vào chi phí kinh doanh thực tế để chọn ngưỡng, không chỉ dựa vào một chỉ số.

### 13. Lưu mô hình
Lưu mô hình Logistic Regression tốt nhất (Logistic Regression Tuned) bằng joblib, sau đó nạp lại file .pkl và kiểm tra dự đoán có khớp với mô hình ban đầu hay không.

In [ ]:
import joblib
import numpy as np
from pathlib import Path

save_dir_lr = Path(RESULTS_DIR) / "Experiment_16_LogisticRegression_tuned"
save_dir_lr.mkdir(parents=True, exist_ok=True)
model_path_lr = save_dir_lr / "best_model.pkl"

# Lưu mô hình tốt nhất
joblib.dump(model_lr_tuned, model_path_lr)
print(f"Đã lưu mô hình tốt nhất tại: {model_path_lr}")

# Nạp lại mô hình và kiểm tra dự đoán
loaded_model_lr = joblib.load(model_path_lr)
sample_pred_lr = loaded_model_lr.predict(X_test[:5])
yes_index_loaded_lr = list(loaded_model_lr.classes_).index(1)
sample_proba_lr = loaded_model_lr.predict_proba(X_test[:5])[:, yes_index_loaded_lr]

assert np.array_equal(
    sample_pred_lr,
    y_pred_lr_tuned[:5],
), "Dự đoán sau khi load mô hình không khớp!"

print("Kiểm tra load mô hình thành công!")
print("5 nhãn dự đoán đầu tiên:", sample_pred_lr)
print("Xác suất lớp Yes tương ứng:", np.round(sample_proba_lr, 4))